# Credit-card fraud classification

Use the pinned data asset from the README, fit the original-size feature and classifier stages, and compare the actual local predictions.

## The quantum kernel is one part of a classical classifier

The visible native overlap qfunc allocates three feature bits, applies the
Pauli embedding and composes a checked adjoint for the other row. FlagQuantum
constructs that coherent kernel. PCA, scaling, t-SNE, kernel-matrix evaluation
and SVM fitting remain classical host work, not computations inferred by synth.
The full archived dataset and original deterministic partitions remain intact.

Try the existing two nonzero feature rows in the preview and inspect the named
zero-state overlap before fitting a model. The full workflow uses shared
quantum.py; propagate embedding edits there. The retained scaler uses the
source's combined transformed partitions, so these metrics are not a strict
training-only preprocessing evaluation or evidence of quantum advantage.


Use the installed optional scientific example environment.

In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QNum
from flagquantum.synth import adjoint
from flagquantum.synth import allocate
from flagquantum.synth import qfunc
from typing import Annotated

from examples.synthesis.applications.finance.credit_card_fraud.main import pauli_kernel
from examples.synthesis.applications.finance.credit_card_fraud.overview.quantum import (
    checked,
)


@qfunc
def overlap_root(
    data1: list[float], data2: list[float], qba: Output[Annotated[QNum, 3]]
) -> None:
    allocate(qba)
    pauli_kernel(data1, qba)
    with adjoint():
        pauli_kernel(data2, qba)


def overlap_preview(left: tuple[float, ...], right: tuple[float, ...]) -> synth.Program:
    return synth.synthesize(
        overlap_root,
        classical_inputs={"data1": checked(left), "data2": checked(right)},
        max_qubits=3,
        max_workspace=0,
        max_operations=96,
        memory_limit_bytes=1024**2,
    )


from importlib.metadata import version


Use local Matplotlib and scikit-learn for the visualizations.

In [ ]:
# See the README for installing the Notebook's scientific packages.


Import the native circuit, data preprocessing and classifier stages.

In [ ]:
import os
from pathlib import Path

import numpy as np
from IPython.display import Image, display

import examples.synthesis.applications.finance.credit_card_fraud.overview.model as model
import examples.synthesis.applications.finance.credit_card_fraud.overview.plots as plots
import examples.synthesis.applications.finance.credit_card_fraud.overview.quantum as quantum
import examples.synthesis.applications.finance.credit_card_fraud.overview.workflow as workflow
from examples.synthesis.applications.finance.credit_card_fraud.overview.main import (
    report,
)

print("Local scikit-learn:", version("scikit-learn"))


Keep the original t-SNE seed for both raw split projections.

In [ ]:
assert model.SEED_TSNE == 20150101
figures = Path("notebook-figures")


Locate the pinned teaching CSV.

In [ ]:
path = Path(os.environ.get("FLAGQUANTUM_CREDIT_CARD_DATA", "creditcard.csv"))


Read every row from the original source asset and apply the preprocessing workflow.

In [ ]:
data = model.prepare(path)
assert (data.rows, data.by_class) == (948, (897, 51))


Keep the original class counts in all three partitions.

In [ ]:
assert model.SPLIT_COUNTS == ((100, 25), (50, 10), (50, 10))


Check both labels in every partition.

In [ ]:
assert all(
    set(group.labels) == {0, 1}
    for group in (data.raw.train, data.raw.test, data.raw.predict)
)


Retain the source's unshuffled within-class order.

In [ ]:
print("Ordered original partitions:", model.SPLIT_COUNTS)


Keep 125 training, 60 testing and 60 prediction rows.

In [ ]:
assert tuple(
    len(group.labels) for group in (data.raw.train, data.raw.test, data.raw.predict)
) == (125, 60, 60)


Exclude the time column and retain all original feature columns.

In [ ]:
assert all(
    group.points.shape[1] == 29
    for group in (data.raw.train, data.raw.test, data.raw.predict)
)


Use the raw data and the original label colors for both t-SNE displays.

In [ ]:
assert model.LABELS == ("Nominal", "Fraud")


Fit and display the original-size training t-SNE projection.

In [ ]:
training_tsne = model.embed(data.raw.train)
assert training_tsne.shape == (125, 2)
training_figure = figures / "training-tsne.png"
plots.scatter(
    training_tsne, data.raw.train.labels, "Credit-card training data", training_figure
)
display(Image(filename=str(training_figure)))


Fit and display the original-size test t-SNE projection.

In [ ]:
test_tsne = model.embed(data.raw.test)
assert test_tsne.shape == (60, 2)
test_figure = figures / "test-tsne.png"
plots.scatter(test_tsne, data.raw.test.labels, "Credit-card test data", test_figure)
display(Image(filename=str(test_figure)))


Encode three transformed features on three qubits.

In [ ]:
assert data.features.train.points.shape == (125, 3)


Fit PCA on all original training rows and transform the three partitions.

In [ ]:
assert data.features.test.points.shape == data.features.predict.points.shape == (60, 3)


Check the finite training-standardized and angularly scaled outputs.

In [ ]:
assert all(
    np.isfinite(group.points).all()
    for group in (data.features.train, data.features.test, data.features.predict)
)


Keep the source's joint min-max map to the interval from minus pi to pi.

In [ ]:
for group in (data.features.train, data.features.test, data.features.predict):
    assert float(np.min(group.points)) >= -np.pi - 1e-12
    assert float(np.max(group.points)) <= np.pi + 1e-12


Pair each transformed feature row with its original label.

In [ ]:
assert all(
    np.array_equal(raw.labels, transformed.labels)
    for raw, transformed in zip(
        (data.raw.train, data.raw.test, data.raw.predict),
        (data.features.train, data.features.test, data.features.predict),
        strict=True,
    )
)


Use two layers and all three pairs of the Z/ZZ feature map.

In [ ]:
left = tuple(map(float, data.features.train.points[0]))
right = tuple(map(float, data.features.train.points[3]))


Build the native inverse-composition overlap on three wires.

In [ ]:
program = overlap_preview(left, right)
assert len(program.registers["qba"]) == 3


Verify two distinct complete overlap states against independent Circuits.

In [ ]:
errors = workflow.compare(data.features)
print(program.report["resources"])


Train from sampled kernel entries and evaluate both actual holdout sets.

In [ ]:
score = workflow.classify(data.features)


Inspect the held-out test accuracy and all predicted labels.

In [ ]:
assert len(score.test_labels) == len(score.predictions) == 60


Run the linear, polynomial, RBF and sigmoid test comparisons.

In [ ]:
classical = workflow.classical(data.features)
assert len(classical) == 4


Report all measured classifications and the prediction accuracy.

In [ ]:
report(workflow.Run(data, score, classical, errors))
